# **Imports**

In [2]:
import os
import yaml
import glob
import pandas as pd

from scipy.stats import ttest_rel
from scipy.stats import wilcoxon

# **Setup**

In [3]:
###################################
# Define Parameters & Settings
###################################

# output directory for deliverables...
output_dir = os.path.join(r'../assets/sgmap-net/classification')
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)

# directory for experiments...
exp_dir = r'../experiments/sgmap-net/classification'

# **Loss Ablation**

## *Loss Strategies*

In [5]:
#############################################
# Class Balance Schemes for Loss Ablation
#############################################

# path to imbalance metrics
stats_path = r'../assets/data_eda/esv1p1_imbalance_metrics.csv'

# read into df
stats = pd.read_csv(stats_path)


##### inverse class frequency - PyTorch implementation for BCE pos_weight - (N-n_c) / n_c
N = len(glob.glob(r'../data/**/patches/*mask.tif'))
n_c = stats['Frequency (n)']
icf = (N-n_c) / n_c


##### IRLbl - just IRLbl
irlbl = stats['IRLbl']


##### ENS - Cui et al. (1/ENS)
raw_w_c = 1 / stats['N_eff']
sum_w_c = raw_w_c.sum()
normalize_factor = 7 / sum_w_c
final_w_c = raw_w_c * normalize_factor

In [6]:
icf

0      0.059684
1      0.649718
2      1.252905
3      1.843601
4      3.046178
5     20.615331
6    113.881481
Name: Frequency (n), dtype: float64

In [7]:
irlbl

0      1.000000
1      1.556802
2      2.126017
3      2.683443
4      3.818289
5     20.397909
6    108.411111
Name: IRLbl, dtype: float64

In [8]:
final_w_c

0    0.142326
1    0.158953
2    0.180176
3    0.202848
4    0.251584
5    1.007628
6    5.056486
Name: N_eff, dtype: float64

## *Loss Ablation Results*

In [70]:


loss_dir = r'../experiments/sgmap-net/classification/loss'


exp_dirs = glob.glob(f"{loss_dir}/**/*", recursive=False)

ablations = []
for d in exp_dirs:
    loss_name = os.path.dirname(d).split(os.sep)[-1]
    input_name = d.split(os.sep)[-1].split('_')[1]
    config_path = os.path.join(d, "config.yml")
    id_path = os.path.join(d, "id_global.csv")
    cd_path = os.path.join(d, "cd_global.csv")

    pos_weight = pd.NA
    if 'pos_weight' in loss_name:
        pos_weight = loss_name.split('-')[1]

    gamma = pd.NA
    alpha = pd.NA
    with open(config_path, "r") as f:
        c = yaml.safe_load(f)
        if 'gamma' in d or 'alpha' in d:
            gamma = c['loss']['classification']['params']['gamma']
            if 'gamma_' in d:
                pos_weight = loss_name.split('_')[1]
        if 'alpha' in d:
            alpha = c['loss']['classification']['params']['alpha']

    id = pd.read_csv(id_path)['macro_f1'].item()
    cd = pd.read_csv(cd_path)['macro_f1'].item()

    df = pd.DataFrame(
        columns=['input', 'pos_weight', 'gamma', 'alpha', 'f1_id', 'f1_cd'], 
        data=[[input_name, pos_weight, gamma, alpha, id, cd]]
        )
    ablations.append(df)

df = pd.concat(ablations)
df['delta'] = df['f1_id'] - df['f1_cd']
df['delta_rel'] = df['delta'] / df['f1_id'] * 100
df['total'] = df['f1_id'] + df['f1_cd']
df.fillna('-', inplace=True)
df.sort_values(['input', 'total'], ascending=[True, False], inplace=True)
output_path = os.path.join(output_dir, 'loss_ablation.csv')
df.to_csv(output_path, index=False)

df

,input,pos_weight,gamma,alpha,f1_id,f1_cd,delta,delta_rel,total
0,dem,-,0.5,0.5,0.680835,0.535058,0.145777,21.411500,1.215892
0,dem,-,3.0,-,0.666824,0.528058,0.138766,20.809985,1.194882
0,dem,-,0.5,0.25,0.662932,0.526220,0.136713,20.622428,1.189152
0,dem,ens,-,-,0.669437,0.508738,0.160699,24.005120,1.178176
0,dem,-,-,-,0.683138,0.492546,0.190591,27.899390,1.175684
0,dem,-,0.5,-,0.659143,0.516193,0.142950,21.687257,1.175335
0,dem,irlbl,0.5,-,0.665752,0.504897,0.160855,24.161396,1.170649
0,dem,-,1.0,-,0.673774,0.489090,0.184684,27.410419,1.162864
0,dem,-,0.5,0.75,0.663911,0.497343,0.166568,25.088953,1.161254
0,dem,icf,-,-,0.658020,0.488861,0.169159,25.707339,1.146880


In [75]:
df.groupby(['pos_weight', 'gamma', 'alpha'])['total'].mean().sort_values(ascending=False)

pos_weight  gamma  alpha
-           0.5    -        1.232564
            3.0    -        1.229897
            0.5    0.75     1.229644
                   0.5      1.229099
                   0.25     1.222793
            -      -        1.220123
            1.0    -        1.218705
            2.0    -        1.216248
            1.5    -        1.213931
irlbl       0.5    -        1.206862
            -      -        1.206665
ens         -      -        1.193714
icf         -      -        1.167348
Name: total, dtype: float64

# **Pre-trained vs. Random Initialization**

In [85]:

random_init_paths = glob.glob(r'../experiments/sgmap-net/classification/random_init/*')

ablations = []
for d in random_init_paths:
    input_name = os.path.split(d)[-1].split('_')[1]
    config_path = os.path.join(d, 'config.yml')
    id_path = os.path.join(d, 'id_global.csv')
    cd_path = os.path.join(d, 'cd_global.csv')

    with open(config_path, "r") as f:
        c = yaml.safe_load(f)
        pt = c['model']['sgmapnet_params']['pretrained']

    id = pd.read_csv(id_path)['macro_f1'].item()
    cd = pd.read_csv(cd_path)['macro_f1'].item()

    df = pd.DataFrame(columns=['input', 'pretrained', 'f1_id', 'f1_cd'], data=[[input_name, pt, id, cd]])
    ablations.append(df)

df = pd.concat(ablations)
df['delta'] = df['f1_id'] - df['f1_cd']
df['delta_rel'] = df['delta'] / df['f1_id'] * 100
df['total'] = df['f1_id'] + df['f1_cd']
df.sort_values(['input', 'total'], ascending=[True, False], inplace=True)
output_path = os.path.join(output_dir, 'pretrain_ablation.csv')
df.to_csv(output_path, index=False)
df

,input,pretrained,f1_id,f1_cd,delta,delta_rel,total
0,dem,True,0.659143,0.516193,0.142950,21.687257,1.175335
0,dem,False,0.650695,0.478100,0.172595,26.524698,1.128795
0,dem+s-ms,True,0.660962,0.601564,0.059398,8.986570,1.262527
0,dem+s-ms,False,0.662831,0.541501,0.121329,18.304710,1.204332
0,s-5,True,0.632307,0.601038,0.031269,4.945165,1.233345
0,s-5,False,0.646918,0.578546,0.068371,10.568794,1.225464
0,s-ms,True,0.644994,0.614055,0.030939,4.796722,1.259049
0,s-ms,False,0.643869,0.583712,0.060156,9.342959,1.227581


# **Single Feature Experiments**

## *Experiments Compilation*

In [ ]:
single_dirs = glob.glob(r'../experiments/sgmap-net/classification/single*/*')

single_experiments = []
for idx, d in enumerate(single_dirs):
    input = os.path.split(d)[1].split('_')[1]
    exp_type = os.path.split(os.path.dirname(d))[1]
    df = pd.DataFrame({'input': input, 'exp_type': exp_type}, index=[0])
    id = pd.read_csv(os.path.join(d, 'id_global.csv'))
    cd = pd.read_csv(os.path.join(d, 'cd_global.csv'))
    df = pd.merge(left=df, right=id, suffixes=['', '_id'], left_index=True, right_index=True)
    df = pd.merge(left=df, right=cd, suffixes=['', '_cd'], left_index=True, right_index=True)
    single_experiments.append(df)

df = pd.concat(single_experiments)
df.sort_values(['input', 'exp_type'], ascending=True, inplace=True)
df.to_csv(r'../assets/sgmap-net/classification/single_compiled.csv', index=False)
df.head(6)

,input,exp_type,macro_precision,macro_recall,macro_f1,auroc,macro_map,micro_accuracy,macro_precision_cd,macro_recall_cd,macro_f1_cd,auroc_cd,macro_map_cd,micro_accuracy_cd
0,dem,single,0.621140,0.708724,0.651104,0.897432,0.696739,0.869606,0.474032,0.615418,0.518696,0.746455,0.566523,0.838728
0,dem,single_sa,0.632110,0.760162,0.671874,0.907449,0.724116,0.876116,0.513544,0.720419,0.571354,0.787741,0.612639,0.815197
0,ep-101,single,0.578217,0.658240,0.614173,0.865997,0.672314,0.859282,0.497633,0.510062,0.480759,0.755963,0.542787,0.832961
0,ep-101,single_sa,0.595247,0.718240,0.644187,0.889977,0.696565,0.862909,0.535245,0.507535,0.513698,0.790178,0.551782,0.853702
0,ep-11,single,0.623125,0.686987,0.649751,0.890832,0.703745,0.869234,0.424184,0.420928,0.406380,0.714870,0.496023,0.836589
0,ep-11,single_sa,0.626235,0.705215,0.648845,0.886033,0.709040,0.866908,0.457603,0.397509,0.411638,0.672123,0.492380,0.844308


## *Comparison*

In [ ]:

single_path = r'../assets/sgmap-net/classification/single_compiled.csv'

df_single = pd.read_csv(single_path)

id = df_single[['input', 'exp_type', 'macro_f1']].pivot(columns='exp_type', index='input', values='macro_f1')
id.rename(columns={'single': 'f1_id', 'single_sa': 'f1_id_sa'}, inplace=True)
id['delta_id'] = id['f1_id'] - id['f1_id_sa']
id['delta_rel_id'] = id['delta_id'] / id['f1_id'] * 100

cd = df_single[['input', 'exp_type', 'macro_f1_cd']].pivot(columns='exp_type', index='input', values='macro_f1_cd')
cd.rename(columns={'single': 'f1_cd', 'single_sa': 'f1_cd_sa'}, inplace=True)
cd['delta_cd'] = cd['f1_cd'] - cd['f1_cd_sa']
cd['delta_rel_cd'] = cd['delta_cd'] / cd['f1_cd'] * 100

df = pd.merge(left=id, right=cd, how='inner', on='input')
df.to_csv(r'../assets/sgmap-net/classification/single_f1_comparison.csv')

df.head()

exp_type,f1_id,f1_id_sa,delta_id,delta_rel_id,f1_cd,f1_cd_sa,delta_cd,delta_rel_cd
input,,,,,,,,
dem,0.651104,0.671874,-0.020770,-3.189917,0.518696,0.571354,-0.052658,-10.151929
ep-101,0.614173,0.644187,-0.030014,-4.886922,0.480759,0.513698,-0.032940,-6.851593
ep-11,0.649751,0.648845,0.000906,0.139473,0.406380,0.411638,-0.005257,-1.293690
ep-201,0.638873,0.657576,-0.018704,-2.927607,0.502631,0.492571,0.010059,2.001364
ep-21,0.640618,0.657497,-0.016878,-2.634700,0.386823,0.413393,-0.026570,-6.868840


In [35]:
#############################################
# Statistical Test if Self-Attention Helps
#############################################

# H0: no attention = self-attention
# HA: no attention < self-attention

##### In-domain...
print('In-domain...')
print('Paired t-test')
id_result = ttest_rel(df['f1_id'], df['f1_id_sa'], alternative="less")
print(id_result.statistic)
print(id_result.pvalue)

print('Wilcoxon...')
id_stat, id_p = wilcoxon(df['f1_id'], df['f1_id_sa'], alternative='less')
print(id_stat)
print(id_p)


##### Cross-domain...
print('\nCross-domain...')
print('Paired t-test')
cd_result = ttest_rel(df['f1_cd'], df['f1_cd_sa'], alternative="less")
print(cd_result.statistic)
print(cd_result.pvalue)

print('Wilcoxon...')
cd_stat, cd_p = wilcoxon(df['f1_cd'], df['f1_cd_sa'], alternative='less')
print(cd_stat)
print(cd_p)

In-domain...
Paired t-test
-2.965991232851052
0.0027431364516770315
Wilcoxon...
138.0
0.0015035983233246952

Cross-domain...
Paired t-test
-0.3699239897810205
0.3568667407624776
Wilcoxon...
314.0
0.4967745130416006


# **Multi-scale Feature Experiments**

## *Experiments Compilation*

In [36]:
ms_dirs = glob.glob(r'../experiments/sgmap-net/classification/multiscale*/*')

ms_experiments = []
for idx, d in enumerate(ms_dirs):
    input = os.path.split(d)[1].split('_')[1]
    exp_type = os.path.split(os.path.dirname(d))[1]
    df = pd.DataFrame({'input': input, 'exp_type': exp_type}, index=[0])
    id = pd.read_csv(os.path.join(d, 'id_global.csv'))
    cd = pd.read_csv(os.path.join(d, 'cd_global.csv'))
    df = pd.merge(left=df, right=id, suffixes=['', '_id'], left_index=True, right_index=True)
    df = pd.merge(left=df, right=cd, suffixes=['', '_cd'], left_index=True, right_index=True)
    ms_experiments.append(df)

df = pd.concat(ms_experiments)
df['input'] = df['input'].apply(lambda x: str(x).replace('-5', '-ms'))
df.sort_values(['input', 'exp_type'], ascending=True, inplace=True)
df.to_csv(r'../assets/sgmap-net/classification/multiscale_compiled.csv', index=False)
df.head(6)

,input,exp_type,macro_precision,macro_recall,macro_f1,auroc,macro_map,micro_accuracy,macro_precision_cd,macro_recall_cd,macro_f1_cd,auroc_cd,macro_map_cd,micro_accuracy_cd
0,ep-ms,multiscale,0.609845,0.727018,0.660042,0.893755,0.703779,0.867001,0.480082,0.361919,0.391306,0.690973,0.463684,0.841146
0,ep-ms,multiscale_concat,0.595923,0.715978,0.628447,0.861839,0.669688,0.842448,0.559470,0.565676,0.545568,0.775533,0.560430,0.853144
0,ep-ms,multiscale_sa,0.601263,0.714099,0.639694,0.888596,0.703401,0.858445,0.466486,0.407747,0.407654,0.706281,0.487518,0.845517
0,plc-ms,multiscale,0.596703,0.626561,0.609333,0.848609,0.662834,0.855562,0.478081,0.611787,0.527573,0.766091,0.560270,0.838635
0,plc-ms,multiscale_concat,0.502914,0.699913,0.567186,0.747149,0.565132,0.770368,0.401822,0.592627,0.463017,0.692731,0.435827,0.743304
0,plc-ms,multiscale_sa,0.565035,0.649718,0.595762,0.849485,0.646218,0.835193,0.465844,0.604924,0.516144,0.749867,0.555515,0.832682


## *Comparison*

In [50]:
ms_path = r'../assets/sgmap-net/classification/multiscale_compiled.csv'

df_ms = pd.read_csv(ms_path)

id = df_ms[['input', 'exp_type', 'macro_f1']].pivot(columns='exp_type', index='input', values='macro_f1')
id.rename(columns={'multiscale': 'f1_id_stack', 'multiscale_sa': 'f1_id_sa', 'multiscale_concat': 'f1_id_concat'}, inplace=True)
id = id[['f1_id_stack', 'f1_id_sa', 'f1_id_concat']]
# id['delta_id'] = id['f1_id_stack'] - id['f1_id_sa']
# id['delta_rel_id'] = id['delta_id'] / id['f1_id'] * 100

cd = df_ms[['input', 'exp_type', 'macro_f1_cd']].pivot(columns='exp_type', index='input', values='macro_f1_cd')
cd.rename(columns={'multiscale': 'f1_cd_stack', 'multiscale_sa': 'f1_cd_sa', 'multiscale_concat': 'f1_cd_concat'}, inplace=True)
cd = cd[['f1_cd_stack', 'f1_cd_sa', 'f1_cd_concat']]
# cd['delta_cd'] = cd['f1_cd'] - cd['f1_cd_sa']
# cd['delta_rel_cd'] = cd['delta_cd'] / cd['f1_cd'] * 100

df = pd.merge(left=id, right=cd, how='inner', on='input')
df.to_csv(r'../assets/sgmap-net/classification/single_f1_comparison.csv')

df.head()

exp_type,f1_id_stack,f1_id_sa,f1_id_concat,f1_cd_stack,f1_cd_sa,f1_cd_concat
input,,,,,,
ep-ms,0.660042,0.639694,0.628447,0.391306,0.407654,0.545568
plc-ms,0.609333,0.595762,0.567186,0.527573,0.516144,0.463017
prc-ms,0.655987,0.658289,0.626108,0.584121,0.587446,0.545888
s-ms,0.634511,0.657652,0.598302,0.608220,0.618954,0.590466
sds-ms,0.630364,0.623147,0.611771,0.588780,0.575102,0.533364


# **Multimodal Experiments**